# Datamine PITRES Process Example

This notebook demonstrates how to configure and run the **`pitres`** process wrapper in `dmstudio`.

## Process Description

## PITRES

Enhanced reserve tabulation of a RESULTS file.

### Input Files:

* **results** (Undefined):
Input results file. In standard format as per [TABRES](<tabres.md>).
Required=Yes

* **model** (Block Model):
Optional input model file to obtain spatial location parameters (such as RL)
Required=No

### Output Files:

### Fields:

* **aux** (Any : RESULTS):
Auxiliary classification field. An additional "rocktype" field. Create separately -
often retrieval criteria are used.
Default=Undefined
Required=No

* **aaux** (Numeric : RESULTS):
Auxiliary area classification field. Creates separate table for each field value,
integers only. Mutually exclusive with **BAUX**
Default=Undefined
Required=No

* **baux** (Numeric : RESULTS):
Auxiliary bench classification field. Creates separate table for each Plane, using
values of this field in each one, integers only. Mutually exclusive with **AAUX**
Default=Undefined
Required=No

* **grade** (Numeric : RESULTS):
Default grade field
Default=Undefined
Required=No

* **row** (Any : RESULTS):
Field to use in place of 'Number' in tables, ie replace RL, East, or North by something
else.
Default=Undefined
Required=No

### Parameters:

* **position**:
Location in cell of reference position. (0) 0 = base of cell 0.5 = centre of cell 1 =
top of cell
Range=0,1
Values=0,0.5,1
Default=0
Required=No

* **tsquash**:
An amount subtracted from tonnes field width [10], +ve or -ve. Default is 0
Range=Undefined
Values=Undefined
Default=0
Required=No

* **gsquash**:
An amount subtracted from grade field width [7], +ve or -ve. Default is 0
Range=Undefined
Values=Undefined
Default=0
Required=No

* **squash**:
An amount subtracted from metal field widths [9
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **gunit**:
Default grade units, in quotes. One of 'g/t', 'ppm', 'oz/t', '%', or 'dwt'
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **munit**:
Default metal units, in quotes. One of 'oz', 'kg', or 'tonn'
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **elemnt**:
Default grade element/compound symbol, in quotes. Maximum of 4 characters
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **tround**:
Rounding control for tonnes field. 1 = round to 10's 2 = 100's 3 = 1000's 4 = use units
of 1000. Default 0 [none]
Range=0,4
Values=0,2,3,4
Default=0
Required=No

* **gdec**:
Decimal places for grade field. 0 = 0 decimal place, 1 = 1 decimal place Default 2
decimal places
Range=0,4
Values=Undefined
Default=2
Required=No

* **mround**:
Rounding control for metal field. 1 = use 0 decimal places for kg units [def 1] 2 =
round to 10's, 3 = 100's 4 = 1000's, Default 0 [none]
Range=0,4
Values=0,1,2,3,4
Default=0
Required=No

* **sdec**:
Decimal places for strip ratio field. 0 = 0 decimal place Default 1 decimal place
Range=Undefined
Values=Undefined
Default=0
Required=No

* **lines**:
Number of lines per page, not including headers. (50)
Range=Undefined
Values=Undefined
Default=50
Required=No

* **width**:
Page width in mm, max of 240.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **numtab**:
Optional table number, 1-99, for display.
Range=1,99
Values=Undefined
Default=Undefined
Required=No

* **volume**:
1 = report in volumes rather than tonnes. (0)
Range=0,1
Values=0,1
Default=0
Required=No

* **tonnesa**:
1 = report individual tonnage components for each grade (ie TonnesA, TonnesB, etc
rather than just overall tonnes of a cell, ie Tonnes. (0)
Range=0,1
Values=0,1
Default=0
Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('pitres')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute pitres
print("Running pitres...")
dm_cmd.pitres(
    results_i=['t_input_file'],  # required
    # model_i='t_mod1',  # optional
    # aux_f='optional',  # optional
    # aaux_f='optional',  # optional
    # baux_f='optional',  # optional
    # grade_f='optional',  # optional
    # row_f='optional',  # optional
    # position_p=0,  # optional
    # tsquash_p=0,  # optional
    # gsquash_p=0,  # optional
    # squash_p='optional',  # optional
    # gunit_p='optional',  # optional
    # munit_p='optional',  # optional
    # elemnt_p='optional',  # optional
    # tround_p=0,  # optional
    # gdec_p=2,  # optional
    # mround_p=0,  # optional
    # sdec_p=0,  # optional
    # lines_p=50,  # optional
    # width_p='optional',  # optional
    # numtab_p='optional',  # optional
    # volume_p=0,  # optional
    # tonnesa_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("pitres execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_pitres_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")